# Συνοδευτικό notebook για το Κεφάλαιο 7
**Φτιάξε το πρώτο σου μεγάλο γλωσσικό μοντέλο — Κεφάλαιο 7: Προετοίμασε τα δεδομένα σου**
Σε αυτό το notebook θα καθαρίσεις τα δεδομένα, θα αφαιρέσεις τα διπλότυπα, θα χωρίσεις τα δεδομένα σε υποσύνολα, θα τεμαχίσεις τα κείμενα και θα τα αποθηκεύσεις σε μορφή JSONL. Θα δεις επίσης μερικά βασικά στατιστικά.
Το μικρό δείγμα δεδομένων περιλαμβάνεται στο notebook· δεν χρειάζεσαι εξωτερικά αρχεία.


Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch07.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch07.ipynb). Τα αγγλικά κείμενα μένουν αυτούσια: ο κατακερματισμός, ο τεμαχισμός και οι εγγραφές του `train.jsonl` εξαρτώνται από τους ακριβείς χαρακτήρες τους.

In [ ]:
# ===== ΕΙΣΑΓΩΓΕΣ =====
import re           # Κανονικές εκφράσεις για αναζήτηση μοτίβων σε κείμενο
import json         # Ανάγνωση και εγγραφή αρχείων JSON και JSONL
import hashlib      # Δημιουργία αποτυπωμάτων για εντοπισμό διπλοτύπων
import unicodedata  # Κανονικοποίηση Unicode (για τον χειρισμό ειδικών χαρακτήρων)
from collections import Counter  # Καταμέτρηση της συχνότητας των λέξεων
import random       # Ανακάτεμα των δεδομένων για διαχωρισμό σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής

print('Η προετοιμασία ολοκληρώθηκε')

## Σύντομος οδηγός εργαλείων της Python

Σε αυτό το notebook θα χρησιμοποιήσεις διάφορα εργαλεία της Python. Δες με μια ματιά πώς λειτουργούν:

**Κανονικές εκφράσεις (regex):** Αναζήτηση και αντικατάσταση σε κείμενο με βάση ένα μοτίβο
- `re.sub(pattern, replacement, text)` — Βρες όλα τα σημεία του κειμένου που ταιριάζουν στο `pattern` και αντικατάστησέ τα με το `replacement`
- Το μοτίβο `<[^>]+>` σημαίνει: βρες ένα `<`, έπειτα έναν ή περισσότερους χαρακτήρες εκτός από `>` και τέλος ένα `>` → ταιριάζει με ετικέτες HTML

**Κατακερματισμός:** Δημιούργησε ένα μοναδικό «αποτύπωμα» για οποιοδήποτε κείμενο
- Ίδιο κείμενο → ίδιο αποτύπωμα (πάντα). Διαφορετικό κείμενο → διαφορετικό αποτύπωμα (σχεδόν πάντα)
- Έτσι εντοπίζεις διπλότυπα χωρίς να συγκρίνεις ολόκληρα έγγραφα
- `hashlib.sha1(text.encode()).hexdigest()` → αποτύπωμα 40 χαρακτήρων

**JSON/JSONL:** Μορφές αποθήκευσης δομημένων δεδομένων
- **JSON:** Ένα μεγάλο αρχείο με όλα τα δεδομένα (πρέπει να φορτώσεις ολόκληρο το αρχείο στη μνήμη)
- **JSONL:** Μία εγγραφή JSON ανά γραμμή (μπορείς να διαβάζεις το αρχείο γραμμή προς γραμμή για να εξοικονομείς μνήμη)

**Σύνολα:** Συλλογές χωρίς διπλότυπα· μπορείς να ελέγξεις γρήγορα αν το X ανήκει στο σύνολο
- `seen = set()` και έπειτα `seen.add(item)` και `item in seen`

**Υποδείξεις τύπων** (`: str`, `: float = 0.8`): Πληροφορίες για όποιον διαβάζει τον κώδικα (η Python δεν επιβάλλει τους τύπους)
- `text: str` σημαίνει «το `text` πρέπει να είναι συμβολοσειρά»
- `train_p: float = 0.8` σημαίνει «το `train_p` πρέπει να είναι αριθμός κινητής υποδιαστολής, με προεπιλεγμένη τιμή 0.8»

## Δείγματα κειμένων
Μερικές σύντομες παράγραφοι για να προσομοιώσεις ένα μικρό σώμα κειμένων.


In [ ]:
raw_docs = [
    "THE TIME MACHINE — CHAPTER I\n\nThis   is   a    sample text… with   odd spacing, smart “quotes”, and tabs\t.",
    "AI systems learn from examples. Data quality shapes model quality.",
    "The key to machine learning is data; the secret to building AI is understanding.",
]
print('Κείμενα:', len(raw_docs))


## Καθαρισμός και κανονικοποίηση

Αφαίρεσε τις ετικέτες HTML και τους χαρακτήρες ελέγχου, αντικατάστησε τις ακολουθίες λευκών χαρακτήρων με ένα κενό και κανονικοποίησε τα εισαγωγικά.


In [ ]:
def clean_text(text: str) -> str:
    """Καθάρισε και κανονικοποίησε το κείμενο για την εκπαίδευση μεγάλων γλωσσικών μοντέλων."""
    # Κανονικοποίηση Unicode: ﬁ → fi, ｆｕｌｌ → full κτλ.
    # NFKC = αποσύνθεση συμβατότητας + κανονική σύνθεση
    text = unicodedata.normalize("NFKC", text)
    
    # Αφαίρεσε τις ετικέτες HTML: <p>, <div>, <span class="foo"> κτλ.
    # Μοτίβο: <, ένας ή περισσότεροι χαρακτήρες εκτός του > και τέλος >
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Αντικατάστησε τις αλλαγές γραμμής και τους στηλοθέτες με κενά
    text = re.sub(r"[\n\t]", " ", text)
    
    # Σύμπτυξε τα διαδοχικά κενά σε ένα και αφαίρεσε τα κενά από την αρχή και το τέλος
    text = re.sub(r"\s+", " ", text).strip()
    
    # Αντικατάστησε τα καμπύλα εισαγωγικά με ίσια
    text = text.replace("“", '"').replace("”", '"')
    
    return text

cleaned_docs = [clean_text(d) for d in raw_docs]
for i, d in enumerate(cleaned_docs):
    print(f"Καθαρισμένο έγγραφο {i}: {d[:80]}...")

## Αφαίρεση διπλοτύπων

Υπολόγισε την τιμή κατακερματισμού κάθε παραγράφου και αφαίρεσε τα διπλότυπα.


In [ ]:
# Δημιούργησε ένα «αποτύπωμα» του κειμένου με τη συνάρτηση κατακερματισμού SHA1
# Ίδιο κείμενο → ίδιο αποτύπωμα (πάντα)
# Διαφορετικό κείμενο → διαφορετικό αποτύπωμα (με συντριπτικά μεγάλη πιθανότητα)
def hash_chunk(text: str) -> str:
    # Η .encode("utf-8") μετατρέπει τη συμβολοσειρά σε byte, όπως απαιτεί η hashlib
    # Η .hexdigest() επιστρέφει την τιμή κατακερματισμού ως συμβολοσειρά 40 χαρακτήρων
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

# Δες πώς λειτουργεί ο κατακερματισμός
sample = "The cat sat on the mat."
print(f"Κείμενο: '{sample}'")
print(f"Τιμή κατακερματισμού: {hash_chunk(sample)}")
print(f"Ίδιο κείμενο, ίδιο αποτύπωμα: {hash_chunk(sample) == hash_chunk(sample)}")
print(f"Διαφορετικό κείμενο, διαφορετικό αποτύπωμα: {hash_chunk(sample) != hash_chunk('Dog.')}")

def dedup_chunks(chunks):
    """Αφαίρεσε τα διπλότυπα τμήματα κειμένου με βάση τα αποτυπώματα κατακερματισμού τους."""
    seen = set()    # Αποθήκευσε τα αποτυπώματα που έχεις ήδη συναντήσει για γρήγορη αναζήτηση
    unique = []     # Κράτησε μόνο τα μοναδικά τμήματα κειμένου
    
    for c in chunks:
        h = hash_chunk(c)
        if h in seen:
            continue        # Παράλειψε το διπλότυπο
        seen.add(h)         # Κατάγραψε το αποτύπωμα
        unique.append(c)    # Κράτησε το τμήμα κειμένου
    
    return unique

# Πρόσθεσε ένα διπλότυπο για να δεις ότι η αφαίρεση διπλοτύπων λειτουργεί
test_docs = cleaned_docs + [cleaned_docs[0]]  # Πρόσθεσε ένα αντίγραφο του πρώτου εγγράφου
deduped = dedup_chunks(test_docs)
print(f'\nΠριν από την αφαίρεση διπλοτύπων: {len(test_docs)} έγγραφα')
print(f'Μετά την αφαίρεση διπλοτύπων:  {len(deduped)} έγγραφα')

## Διαχωρισμός σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής (ανά έγγραφο)

Κράτησε τα συναφή αποσπάσματα στο ίδιο σύνολο για να αποφύγεις τη διαρροή δεδομένων μεταξύ συνόλων.


In [ ]:
def split_docs(docs: list, train_p: float = 0.8, val_p: float = 0.1, seed: int = 42):
    """Χώρισε τα έγγραφα σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής.
    
    Παράμετροι:
        docs: Λίστα εγγράφων προς διαχωρισμό
        train_p: Αναλογία για εκπαίδευση (προεπιλογή: 0.8 = 80%)
        val_p: Αναλογία για επικύρωση (προεπιλογή: 0.1 = 10%)
        seed: Τιμή αρχικοποίησης της γεννήτριας τυχαίων αριθμών για αναπαραγώγιμα αποτελέσματα
    
    Επιστρέφει:
        Τις λίστες train, val, test (το υπόλοιπο αναλογεί στη test)
    """
    # Δούλεψε σε αντίγραφο για να μην τροποποιήσεις την αρχική λίστα
    docs = list(docs)
    random.seed(seed)
    random.shuffle(docs)
    
    n = len(docs)
    n_train = int(n * train_p)
    n_val = int(n * val_p)
    
    train = docs[:n_train]
    val = docs[n_train:n_train + n_val]
    test = docs[n_train + n_val:]
    
    # Σε πολύ μικρά σώματα κειμένων, η στρογγυλοποίηση μπορεί να αφήσει ένα σύνολο κενό
    # (με 3 έγγραφα και αναλογίες 0.34/0.33, η val παίρνει int(3*0.33)=0 έγγραφα).
    # Αν κάποιο σύνολο έμεινε κενό, μετάφερε ένα έγγραφο από το μεγαλύτερο σύνολο,
    # ώστε κάθε σύνολο να έχει τουλάχιστον ένα έγγραφο, εφόσον αυτό είναι εφικτό.
    if n >= 3:
        for target in (train, val, test):
            if not target:
                donors = sorted((train, val, test), key=len, reverse=True)
                if len(donors[0]) > 1:
                    target.append(donors[0].pop())

    return train, val, test

# Χώρισε τα έγγραφα μετά την αφαίρεση των διπλοτύπων
train_docs, val_docs, test_docs = split_docs(deduped, train_p=0.6, val_p=0.2, seed=42)
print(f'Εκπαίδευση: {len(train_docs)}, Επικύρωση: {len(val_docs)}, Δοκιμή: {len(test_docs)}')


## Κατάτμηση κειμένου για παράθυρα συμφραζομένων
Χώρισε τα μεγάλα κείμενα σε επικαλυπτόμενα τμήματα, ώστε να διατηρούνται τα συμφραζόμενα στα όρια των τμημάτων.


In [ ]:
def chunk_text(text: str, max_chars: int = 200, overlap: int = 50):
    """Χώρισε το κείμενο σε επικαλυπτόμενα τμήματα.
    
    Ορίσματα:
        text: Κείμενο προς τεμαχισμό
        max_chars: Μέγιστος αριθμός χαρακτήρων ανά τμήμα
        overlap: Αριθμός κοινών χαρακτήρων μεταξύ διαδοχικών τμημάτων
    
    Επιστρέφει:
        Λίστα με τα τμήματα του κειμένου
    """
    assert 0 <= overlap < max_chars, "Η τιμή του overlap πρέπει να είναι μικρότερη από την τιμή του max_chars"

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break  # Το τμήμα έφτασε στο τέλος του κειμένου· δεν χρειάζεται άλλο τμήμα.
        start += max_chars - overlap  # Μετακίνησε την αρχή κατά (max_chars - overlap) χαρακτήρες.
    return chunks

# Χώρισε σε τμήματα τα κείμενα όλων των υποσυνόλων.
chunked = []
for split, docs in [('train', train_docs), ('val', val_docs), ('test', test_docs)]:
    for d in docs:
        for c in chunk_text(d, max_chars=120, overlap=30):
            chunked.append({'text': c, 'split': split, 'source': 'toy'})
            
print('Συνολικός αριθμός τμημάτων:', len(chunked))


## Δες την επικάλυψη
Δες πώς η επικάλυψη διατηρεί τα συμφραζόμενα στα όρια των τμημάτων.


In [ ]:
# Δημιούργησε ένα κείμενο δοκιμής με γνωστές θέσεις χαρακτήρων.
test_text = "A" * 500  # 500 χαρακτήρες
chunks = chunk_text(test_text, max_chars=200, overlap=50)

print(f"Μήκος κειμένου: {len(test_text)}")
print(f"Αριθμός τμημάτων: {len(chunks)}")
print(f"Μήκη τμημάτων: {[len(c) for c in chunks]}")

# Έλεγξε την επικάλυψη μεταξύ διαδοχικών τμημάτων.
# Η σύνταξη των τομών στην Python:
#   text[-50:]  = οι τελευταίοι 50 χαρακτήρες (ο αρνητικός δείκτης μετρά από το τέλος)
#   text[:50]   = οι πρώτοι 50 χαρακτήρες
if len(chunks) >= 2:
    # Οι τελευταίοι 50 χαρακτήρες του τμήματος 0 πρέπει να είναι ίδιοι με τους πρώτους 50 του τμήματος 1.
    chunk0_end = chunks[0][-50:]    # Οι τελευταίοι 50 χαρακτήρες του τμήματος 0
    chunk1_start = chunks[1][:50]   # Οι πρώτοι 50 χαρακτήρες του τμήματος 1
    overlap_matches = chunk0_end == chunk1_start
    
    print(f"\nΤαύτιση στην περιοχή επικάλυψης: {overlap_matches}")
    print(f"Το τμήμα 0 τελειώνει με: ...{chunks[0][-10:]}")
    print(f"Το τμήμα 1 αρχίζει με: {chunks[1][:10]}...")
    
# Δοκίμασε με πραγματικό κείμενο.
real_text = "This is sentence one. This is sentence two. This is sentence three." * 5
real_chunks = chunk_text(real_text, max_chars=100, overlap=30)
print(f"\nΤο πραγματικό κείμενο χωρίστηκε σε {len(real_chunks)} τμήματα")
print(f"Τμήμα 0: ...{real_chunks[0][-40:]}")
print(f"Τμήμα 1: {real_chunks[1][:40]}...")
print("\n✅ Η επικάλυψη βοηθά να διατηρούνται ολόκληρες οι προτάσεις που κόβονται στα όρια των τμημάτων, αλλά δεν το εγγυάται.")

## Έλεγχοι ποιότητας και ορθότητας
Εντόπισε έγκαιρα προβλήματα με αυτοματοποιημένους ελέγχους που επισημαίνουν κενά τμήματα, υπολείμματα HTML και ασυνήθιστα μεγέθη.


In [ ]:
def sanity_check(chunks, stage_name):
    """Έλεγξε την ορθότητα των δεδομένων σε οποιοδήποτε στάδιο της επεξεργασίας."""
    print(f"\n{'='*50}")
    print(f"Έλεγχος ορθότητας: {stage_name}")
    print(f"{'='*50}")
    
    if not chunks:
        print("⚠️  ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Δεν υπάρχουν τμήματα!")
        return
    
    # Βασικά στατιστικά στοιχεία
    print(f"✓ Σύνολο τμημάτων: {len(chunks)}")
    lengths = [len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks]
    avg_len = sum(lengths) / len(lengths)
    print(f"✓ Μέσο μήκος: {avg_len:.0f}")
    print(f"✓ Μέγιστο μήκος: {max(lengths)}")
    print(f"✓ Ελάχιστο μήκος: {min(lengths)}")
    
    # Έλεγξε για προβλήματα
    if max(lengths) > 10 * avg_len:
        print("⚠️  ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Το μέγιστο μήκος ξεπερνά το δεκαπλάσιο του μέσου μήκους — ίσως υπάρχει πρόβλημα στην κατάτμηση")
    
    # Αναζήτησε ενδείξεις HTML: π.χ. η λέξη dividend περιέχει το div χωρίς να είναι ετικέτα
    texts = [c if isinstance(c, str) else c.get('text', '') for c in chunks]
    all_text = ' '.join(texts).lower()
    html_words = {'div', 'span', 'href', 'html', 'class', 'src'}
    found_html = [w for w in html_words if w in all_text]
    if found_html:
        print(f"⚠️  ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Έλεγξε τα πιθανά ίχνη HTML: {found_html}")
    else:
        print("✓ Δεν βρέθηκε καμία από τις συμβολοσειρές που αναζητήθηκαν")
    
    # Έλεγξε για κενά τμήματα
    empty = sum(1 for l in lengths if l < 10)
    if empty > 0:
        print(f"⚠️  ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Τμήματα με λιγότερους από 10 χαρακτήρες: {empty}")
    else:
        print("✓ Δεν υπάρχουν κενά τμήματα")
    
    # Δείγμα
    sample = texts[0] if texts else "N/A"
    print(f"\n✓ Δείγμα: {sample[:100]}...")
    print()

# Έλεγξε τα δεδομένα μετά την κατάτμηση
sanity_check(chunked, "Μετά την κατάτμηση")

# Μπορείς να εκτελέσεις τον έλεγχο μετά από κάθε στάδιο:
# sanity_check(cleaned_docs, "Μετά τον καθαρισμό")
# sanity_check(deduped, "Μετά την αφαίρεση διπλοτύπων")


## Αναλυτικό παράδειγμα: Η διαδικασία επεξεργασίας από την αρχή ως το τέλος
Ακολούθησε όλα τα βήματα, από τα ακατέργαστα έγγραφα με HTML και διπλότυπα μέχρι τα δεδομένα που είναι έτοιμα για αποθήκευση σε μορφή JSONL.


In [ ]:
print("="*60)
print("ΟΛΗ Η ΔΙΑΔΙΚΑΣΙΑ ΕΠΕΞΕΡΓΑΣΙΑΣ ΔΕΔΟΜΕΝΩΝ ΒΗΜΑ ΒΗΜΑ")
print("="*60)

# Βήμα 1: Ξεκίνα με ανεπεξέργαστα έγγραφα (με διπλότυπα, HTML και ακανόνιστα κενά)
print("\n📥 ΒΗΜΑ 1: Ανεπεξέργαστα έγγραφα")
raw_pipeline_docs = [
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc1"},
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc2"},  # ακριβές αντίγραφο!
    {"text": "<p>The dog    ran\tin the park.</p>", "source": "doc3"},
    {"text": "The bird flew over the house.", "source": "doc4"}
]
print(f"   Ανεπεξέργαστα έγγραφα: {len(raw_pipeline_docs)}")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text'][:50]}...")

# Βήμα 2: Καθάρισε κάθε έγγραφο
print("\n🧹 ΒΗΜΑ 2: Καθαρισμός")
for doc in raw_pipeline_docs:
    doc["text"] = clean_text(doc["text"])
print("   Αφαιρέθηκαν οι ετικέτες HTML και κανονικοποιήθηκαν τα κενά")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text']}")

# Βήμα 3: Αφαίρεσε τα διπλότυπα και κράτησε την ετικέτα προέλευσης για κάθε μοναδικό έγγραφο
print("\n🔍 ΒΗΜΑ 3: Αφαίρεση διπλοτύπων")
unique_pipeline, seen_pipeline = [], set()
for doc in raw_pipeline_docs:
    h = hash_chunk(doc["text"])
    if h in seen_pipeline:
        continue
    seen_pipeline.add(h)
    unique_pipeline.append(doc)
print(f"   Πριν: {len(raw_pipeline_docs)} έγγραφα")
print(f"   Μετά: {len(unique_pipeline)} μοναδικά έγγραφα")
for i, doc in enumerate(unique_pipeline):
    print(f"   {i+1}. ({doc['source']}) {doc['text']}")

# Βήμα 4: Χώρισε τα έγγραφα σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής
# (Με μόλις 3 έγγραφα, οι αναλογίες αυτές δίνουν τουλάχιστον 1 έγγραφο ανά σύνολο)
print("\n📊 ΒΗΜΑ 4: Διαχωρισμός σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής")
train_p, val_p, test_p = split_docs(unique_pipeline, train_p=0.34, val_p=0.33, seed=42)
for split_name, split_list in [("train", train_p), ("val", val_p), ("test", test_p)]:
    print(f"   {split_name}: {[d['source'] for d in split_list]}")

# Βήμα 5: Χώρισε τα έγγραφα σε τμήματα ανά σύνολο, ώστε να μην μπερδευτούν οι ετικέτες των συνόλων
# (Μικρή τιμή max_chars για το παράδειγμα· για πραγματικά έγγραφα η τιμή είναι 800 ή μεγαλύτερη)
print("\n✂️  ΒΗΜΑ 5: Τεμαχισμός κειμένου")
final_records = []
for split_name, split_list in [("train", train_p), ("val", val_p), ("test", test_p)]:
    for doc in split_list:
        for chunk in chunk_text(doc["text"], max_chars=50, overlap=10):
            final_records.append({"text": chunk, "source": doc["source"],
                                  "split": split_name, "length": len(chunk)})
print(f"   Μετά τον τεμαχισμό: {len(final_records)} εγγραφές "
      f"(εκπαίδευση={sum(r['split']=='train' for r in final_records)}, "
      f"επικύρωση={sum(r['split']=='val' for r in final_records)}, "
      f"δοκιμή={sum(r['split']=='test' for r in final_records)})")

# Βήμα 6: Αυτές είναι οι εγγραφές που θα χωρίσεις σε token στο Κεφάλαιο 8. Κάθε εγγραφή διατηρεί
# τις ετικέτες προέλευσης και συνόλου που παίρνει κατά τον τεμαχισμό.
print("\n💾 ΒΗΜΑ 6: Προετοιμασία για JSONL")
print(f"   Έτοιμες για αποθήκευση: {len(final_records)} εγγραφές")
print(f"   Παράδειγμα εγγραφής: {final_records[0]}")

print("\n✅ Η ΕΠΕΞΕΡΓΑΣΙΑ ΟΛΟΚΛΗΡΩΘΗΚΕ!")
print(f"   Ξεκίνησες με {len(raw_pipeline_docs)} ανεπεξέργαστα έγγραφα (με ένα διπλότυπο)")
print(f"   Κατέληξες σε {len(final_records)} καθαρές εγγραφές JSONL χωρίς διπλότυπα")
print(f"   Οι τρεις σύντομες εγγραφές δείχνουν μόνο τη ΜΟΡΦΗ του αρχείου·")
print(f"   δεν αποτελούν σώμα κειμένων για εκπαίδευση. Στο Κεφάλαιο 8 θα τις χωρίσεις σε token.")


In [ ]:
def save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Εξήγαγε τα σύνολα του παραδείγματος διατηρώντας για κάθε εγγραφή την προέλευση και το σύνολό της.
# Το αρχείο train.jsonl που δημιουργείται εδώ είναι αυτό που φορτώνει το σημειωματάριο του Κεφαλαίου 8.
save_jsonl(final_records, 'train.jsonl')
print('Δημιουργήθηκε το train.jsonl με', len(final_records), 'εγγραφές')
for r in final_records:
    print(' ', r)


## Συνοπτικά στατιστικά

Ποσοστό διπλοτύπων, συχνότερες λέξεις και ενδεικτικές εγγραφές.


In [ ]:
def duplicate_rate(texts):
    hashes = [hash_chunk(t) for t in texts]
    return 1 - (len(set(hashes)) / len(hashes))

def top_words(texts, k=10):
    words = " ".join(texts).lower().split()
    return Counter(words).most_common(k)

texts_all = [r['text'] for r in chunked]
print('Αναλογία διπλοτύπων:', duplicate_rate(texts_all))
print('Συχνότερες λέξεις:', top_words(texts_all, k=8))
print('Ενδεικτικές εγγραφές:', chunked[:2])


## Σύνοψη

Σε αυτό το σημειωματάριο έμαθες πώς να προετοιμάζεις δεδομένα για την εκπαίδευση μεγάλων γλωσσικών μοντέλων (LLM), από την αρχή μέχρι το τέλος:

**1. Καθαρισμός κειμένου:** Αφαίρεσε τη σήμανση HTML, κανονικοποίησε τους χαρακτήρες κενού και χειρίσου τους ειδικούς χαρακτήρες
**2. Αφαίρεση διπλοτύπων:** Χρησιμοποίησε κατακερματισμό για να εντοπίσεις και να αφαιρέσεις τα ακριβή αντίγραφα
**3. Διαχωρισμός σε σύνολα εκπαίδευσης, επικύρωσης και δοκιμής:** Χώρισε τα δεδομένα ανά έγγραφο για να αποφύγεις τη διαρροή δεδομένων μεταξύ των συνόλων
**4. Κατάτμηση με επικάλυψη:** Χώρισε τα μεγάλα κείμενα σε τμήματα με μέγεθος κατάλληλο για το LLM, διατηρώντας τα συμφραζόμενα
**5. Έλεγχοι ποιότητας:** Οι αυτοματοποιημένοι έλεγχοι εντοπίζουν έγκαιρα πιθανά προβλήματα
**6. Μορφή JSONL:** Αποθήκευσε τα δεδομένα σε μορφή που σου επιτρέπει να τα διαβάζεις εγγραφή προς εγγραφή

**Βασικές έννοιες:**
- Η **επικάλυψη** διατηρεί τα συμφραζόμενα στα όρια των τμημάτων (περιορίζει το πρόβλημα των κομμένων προτάσεων, χωρίς να εγγυάται ότι κάθε πρόταση θα παραμείνει ολόκληρη)
- Ο **κατακερματισμός** παράγει γρήγορα ντετερμινιστικά αποτυπώματα για τον εντοπισμό διπλοτύπων
- Ο **διαχωρισμός ανά έγγραφο** κρατά όλα τα τμήματα του ίδιου εγγράφου στο ίδιο σύνολο
- Οι **έλεγχοι ποιότητας** εντοπίζουν υπολείμματα HTML, κενά τμήματα και ασυνήθιστα μεγέθη τμημάτων πριν προκαλέσουν προβλήματα στην εκπαίδευση

**Επόμενα βήματα:**
- Για το Κεφάλαιο 8 κράτησε το `train.jsonl`. Παρά το όνομά του, περιέχει και τα τρία σύνολα (`train`, `val`, `test`). Στο Colab κατέβασέ το πριν κλείσεις το περιβάλλον του Κεφαλαίου 7: `from google.colab import files; files.download('train.jsonl')`.
- Εφάρμοσε τη διαδικασία σε πραγματικά δεδομένα μεγαλύτερης κλίμακας (Βικιπαίδεια, Common Crawl, βιβλία)
- Δοκίμασε διαφορετικές τιμές επικάλυψης για τη δική σου εφαρμογή

Η προετοιμασία των δεδομένων είναι το θεμέλιο κάθε καλού LLM!
